# Experiment B — Reconstructed acceleration → frozen raw-trained CNN

**Goal:** test whether the reconstructed acceleration preserves the information used by the existing raw-acceleration CNN, and whether the held-out-user embedding geometry becomes better or worse.

This notebook implements the strict Experiment B protocol:

```text
Raw train users
    ↓
train CNN once (already done in the baseline notebook)
    ↓
freeze checkpoint + class mapping + split + raw-train normalization
    ↓
held-out test raw ─────────────→ frozen CNN → raw reference metrics
held-out test reconstruction ─→ frozen CNN → Experiment B metrics
```

Important controls:

- **No CNN retraining on reconstruction.**
- **No normalization re-estimation on reconstruction.** The mean/std are loaded from the raw-trained checkpoint.
- **Same train / validation / test users** as the raw baseline checkpoint.
- **Same samples, labels, valid lengths and valid masks.**
- Reconstruction is expected to be the standalone `(S, T, 3)` `*_padded_reconstructed_accel_m_s2.npy` generated next to each padded package.
- Retrieval/prototype/linear-probe diagnostics use **raw-train embeddings as the reference**, so they do not adapt to the reconstruction distribution.

The notebook reports both:
1. **information preservation / downstream performance**, and
2. **class geometry / cluster quality**.

## 1. Imports and repository helpers

In [ ]:
from __future__ import annotations

import copy
import json
import math
import random
import sys
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Mapping, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from torch import nn
from torch.utils.data import DataLoader, Dataset, TensorDataset


def locate_repository_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "snn").is_dir():
            return candidate.resolve()

    try:
        import snn as installed_snn
    except ModuleNotFoundError as error:
        raise RuntimeError(
            "Could not locate the WritingRing repository. Start Jupyter from the repository "
            'or install it with `python -m pip install -e ".[snn,notebook]"`.'
        ) from error
    return Path(installed_snn.__file__).resolve().parent.parent


REPOSITORY_ROOT = locate_repository_root()
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

import snn  # noqa: E402
from snn.action0_dataset import (  # noqa: E402
    Action0DatasetError,
    PADDING_DATASET_SUMMARY_FILENAME,
    SPIKE_IMU_CHANNEL_COUNT,
    load_padding_dataset_metadata,
)
from snn.action0_engine import _classification_metrics  # noqa: E402
from snn.train_action0 import _resolve_device, _set_random_seed, _validate_user_splits  # noqa: E402
from writingring.segment_padding import SPIKE_IMU_FEATURE_SCHEMA  # noqa: E402

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)

print(f"Repository root: {REPOSITORY_ROOT}")
print(f"snn module:      {Path(snn.__file__).resolve()}")
print(f"PyTorch:         {torch.__version__}")

## 2. Experiment configuration

Edit `ROOT` and, if needed, `BASELINE_CHECKPOINT_PATH`.

`ROOT` may point to the dataset-combination root or directly to its `segmentation_padded/` directory.

In [ ]:
# ---------------------------------------------------------------------
# Dataset and baseline checkpoint
# ---------------------------------------------------------------------
ROOT = Path("outputs/action0_rectified/low-pass/aligned-board-events")

BASELINE_CHECKPOINT_PATH = (
    REPOSITORY_ROOT
    / "notebooks"
    / "artifacts"
    / "acceleration_cnn_representation"
    / "best_acceleration_cnn.pt"
)

# ---------------------------------------------------------------------
# Runtime
# ---------------------------------------------------------------------
BATCH_SIZE = 128
NUM_WORKERS = 0
USE_GPU = True
RANDOM_SEED = 12345

# ---------------------------------------------------------------------
# Representation evaluation
# ---------------------------------------------------------------------
K_VALUES = (1, 3, 5, 10, 20, 50)
KNN_K_CANDIDATES = (1, 3, 5, 7, 10, 15, 20)
SIMILARITY_QUERY_BATCH_SIZE = 128

LINEAR_PROBE_EPOCHS = 200
LINEAR_PROBE_LEARNING_RATE = 1e-2
LINEAR_PROBE_WEIGHT_DECAY = 1e-4
LINEAR_PROBE_BATCH_SIZE = 512
LINEAR_PROBE_PATIENCE = 30

KMEANS_N_INIT = 10
KMEANS_MAX_ITER = 100
SILHOUETTE_MAX_SAMPLES = 3000
PCA_MAX_SAMPLES = 5000

# ---------------------------------------------------------------------
# Outputs
# ---------------------------------------------------------------------
OUTPUT_DIR = (
    REPOSITORY_ROOT
    / "notebooks"
    / "artifacts"
    / "experiment_B_reconstruction_frozen_cnn"
)
SAVE_ARTIFACTS = True
SAVE_FIGURES = True

ACCELERATION_SLICE = slice(15, 18)
ACCELERATION_CHANNEL_NAMES = (
    "acceleration_x_m_s2",
    "acceleration_y_m_s2",
    "acceleration_z_m_s2",
)

RECONSTRUCTION_SUFFIX = "_padded_reconstructed_accel_m_s2.npy"
RECONSTRUCTION_METADATA_SUFFIX = "_padded_reconstructed_accel_metadata.json"

if BATCH_SIZE <= 0:
    raise ValueError("BATCH_SIZE must be positive")
if any(k <= 0 for k in (*K_VALUES, *KNN_K_CANDIDATES)):
    raise ValueError("All K values must be positive")

print(f"Configured ROOT:          {ROOT}")
print(f"Baseline checkpoint:      {BASELINE_CHECKPOINT_PATH}")
print(f"Experiment B output dir:  {OUTPUT_DIR}")

## 3. Resolve padded packages and validate reconstruction alignment

For every canonical `*_paddedSpikeIMU.npy`, this loader requires:

- the original padded SpikeIMU package,
- labels,
- `valid_lengths`,
- `valid_mask`,
- padding manifest / summary,
- the reconstruction `*_padded_reconstructed_accel_m_s2.npy`,
- reconstruction metadata JSON.

The reconstruction must have the exact same `(segment, time)` axes as the padded raw package and exactly three acceleration channels.

In [ ]:
def natural_key(text: str) -> tuple[object, ...]:
    parts: list[object] = []
    token = ""
    numeric = False
    for char in str(text):
        current_numeric = char.isdigit()
        if token and current_numeric != numeric:
            parts.append(int(token) if numeric else token.lower())
            token = ""
        token += char
        numeric = current_numeric
    if token:
        parts.append(int(token) if numeric else token.lower())
    return tuple(parts)


def load_json_object(path: Path) -> dict[str, object]:
    try:
        value = json.loads(path.read_text(encoding="utf-8"))
    except (OSError, UnicodeError, json.JSONDecodeError) as error:
        raise Action0DatasetError(f"Could not read JSON object {path}: {error}") from error
    if not isinstance(value, dict):
        raise Action0DatasetError(f"Expected JSON object: {path}")
    return value


def parse_bool_value(value: object, *, context: str) -> bool:
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    text = str(value).strip().lower()
    if text in {"true", "1", "yes"}:
        return True
    if text in {"false", "0", "no"}:
        return False
    raise Action0DatasetError(f"{context}: expected boolean, got {value!r}")


def parse_manifest_int(value: object, *, field: str, path: Path) -> int:
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "null"}:
        raise Action0DatasetError(f"{path}: required integer field {field!r} is empty")
    try:
        return int(text, 10)
    except ValueError:
        pass
    try:
        number = float(text)
    except ValueError as error:
        raise Action0DatasetError(
            f"{path}: field {field!r} must be integer-valued, got {value!r}"
        ) from error
    if not math.isfinite(number) or not number.is_integer():
        raise Action0DatasetError(
            f"{path}: field {field!r} must be integer-valued, got {value!r}"
        )
    return int(number)


def resolve_padded_root(root: Path) -> Path:
    root = Path(root).expanduser()
    if not root.is_absolute():
        root = REPOSITORY_ROOT / root
    root = root.resolve()

    direct_summary = root / PADDING_DATASET_SUMMARY_FILENAME
    if direct_summary.is_file():
        return root

    child = root / "segmentation_padded"
    if (child / PADDING_DATASET_SUMMARY_FILENAME).is_file():
        return child

    candidates = sorted(root.rglob(PADDING_DATASET_SUMMARY_FILENAME)) if root.is_dir() else []
    if len(candidates) == 1:
        return candidates[0].parent
    if not candidates:
        raise FileNotFoundError(
            f"Could not find {PADDING_DATASET_SUMMARY_FILENAME!r} below {root}."
        )
    candidate_text = "\n".join(f"  - {path.parent}" for path in candidates)
    raise Action0DatasetError(
        "ROOT resolves to multiple padded datasets; select one exact segmentation_padded root:\n"
        + candidate_text
    )


@dataclass(slots=True)
class PairedPackage:
    user: str
    action: str
    stem: str
    directory: Path
    padded_spike_imu_path: Path
    reconstructed_acceleration_path: Path
    reconstruction_metadata_path: Path
    labels_path: Path
    valid_lengths_path: Path
    valid_mask_path: Path
    padding_manifest_path: Path
    padding_summary_path: Path
    padded_spike_imu: np.ndarray
    reconstructed_acceleration: np.ndarray
    labels: np.ndarray
    valid_lengths: np.ndarray
    valid_mask: np.ndarray

    @property
    def segment_count(self) -> int:
        return int(self.padded_spike_imu.shape[0])

    @property
    def padded_length(self) -> int:
        return int(self.padded_spike_imu.shape[1])


def discover_padded_spike_paths(padded_root: Path) -> tuple[Path, ...]:
    paths = sorted(
        padded_root.rglob("*_paddedSpikeIMU.npy"),
        key=lambda path: tuple(natural_key(part) for part in path.relative_to(padded_root).parts),
    )
    if not paths:
        raise Action0DatasetError(f"No *_paddedSpikeIMU.npy files found below {padded_root}")
    return tuple(paths)


def assert_finite_memmap(values: np.ndarray, *, path: Path, chunk_segments: int = 256) -> None:
    if not np.issubdtype(values.dtype, np.number):
        raise Action0DatasetError(f"Expected numeric array: {path}; dtype={values.dtype}")
    for start in range(0, len(values), chunk_segments):
        stop = min(start + chunk_segments, len(values))
        if not np.isfinite(values[start:stop]).all():
            raise Action0DatasetError(
                f"Non-finite values found in {path}, segments {start}:{stop}"
            )


def load_and_validate_paired_package(
    padded_path: Path,
    *,
    padded_root: Path,
    root_target_length: int,
) -> tuple[PairedPackage, pd.DataFrame]:
    action_dir = padded_path.parent
    user_dir = action_dir.parent
    if not user_dir.name.startswith("user_") or not action_dir.name.startswith("action_"):
        raise Action0DatasetError(
            f"Expected <root>/user_*/action_* hierarchy, got {padded_path}"
        )

    user = user_dir.name
    action = action_dir.name.removeprefix("action_")
    stem = padded_path.name.removesuffix("_paddedSpikeIMU.npy")
    expected_stem = f"{user}_action_{action}"
    if stem != expected_stem:
        raise Action0DatasetError(
            f"Package stem mismatch in {action_dir}: expected {expected_stem!r}, got {stem!r}"
        )

    labels_path = action_dir / f"{stem}_labels.npy"
    valid_lengths_path = action_dir / f"{stem}_valid_lengths.npy"
    valid_mask_path = action_dir / f"{stem}_valid_mask.npy"
    padding_manifest_path = action_dir / f"{stem}_padding_manifest.csv"
    padding_summary_path = action_dir / f"{stem}_padding_summary.json"
    reconstructed_path = action_dir / f"{stem}{RECONSTRUCTION_SUFFIX}"
    reconstruction_metadata_path = action_dir / f"{stem}{RECONSTRUCTION_METADATA_SUFFIX}"

    required = (
        labels_path,
        valid_lengths_path,
        valid_mask_path,
        padding_manifest_path,
        padding_summary_path,
        reconstructed_path,
        reconstruction_metadata_path,
    )
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError(
            f"Missing required Experiment B files for {user}/action_{action}: "
            + ", ".join(missing)
        )

    padded_spike = np.load(padded_path, allow_pickle=False, mmap_mode="r")
    reconstructed = np.load(reconstructed_path, allow_pickle=False, mmap_mode="r")
    labels = np.load(labels_path, allow_pickle=False).astype(str)
    valid_lengths_raw = np.load(valid_lengths_path, allow_pickle=False)
    valid_mask = np.load(valid_mask_path, allow_pickle=False, mmap_mode="r")

    if not np.issubdtype(valid_lengths_raw.dtype, np.integer):
        raise Action0DatasetError(
            f"{valid_lengths_path}: valid_lengths must have integer dtype"
        )
    valid_lengths = valid_lengths_raw.astype(np.int64, copy=False)

    if padded_spike.ndim != 3 or padded_spike.shape[2] != SPIKE_IMU_CHANNEL_COUNT:
        raise Action0DatasetError(
            f"{padded_path}: expected (S,T,{SPIKE_IMU_CHANNEL_COUNT}), got {padded_spike.shape}"
        )
    segment_count, padded_length, _ = map(int, padded_spike.shape)
    if padded_length != int(root_target_length):
        raise Action0DatasetError(
            f"{padded_path}: T={padded_length} != root target_length={root_target_length}"
        )
    if reconstructed.shape != (segment_count, padded_length, 3):
        raise Action0DatasetError(
            f"{reconstructed_path}: expected {(segment_count, padded_length, 3)}, "
            f"got {reconstructed.shape}"
        )
    if labels.shape != (segment_count,):
        raise Action0DatasetError(
            f"{labels_path}: expected {(segment_count,)}, got {labels.shape}"
        )
    if valid_lengths.shape != (segment_count,):
        raise Action0DatasetError(
            f"{valid_lengths_path}: expected {(segment_count,)}, got {valid_lengths.shape}"
        )
    if valid_mask.shape != (segment_count, padded_length):
        raise Action0DatasetError(
            f"{valid_mask_path}: expected {(segment_count, padded_length)}, got {valid_mask.shape}"
        )
    if valid_mask.dtype != np.dtype(np.bool_):
        raise Action0DatasetError(f"{valid_mask_path}: valid_mask must have bool dtype")
    if np.any(valid_lengths <= 0) or np.any(valid_lengths > padded_length):
        raise Action0DatasetError(
            f"{valid_lengths_path}: valid lengths must lie in [1,{padded_length}]"
        )

    assert_finite_memmap(padded_spike, path=padded_path)
    assert_finite_memmap(reconstructed, path=reconstructed_path)

    time_index = np.arange(padded_length, dtype=np.int64)[None, :]
    for start in range(0, segment_count, 1024):
        stop = min(start + 1024, segment_count)
        mask_chunk = np.asarray(valid_mask[start:stop], dtype=np.bool_)
        expected_mask = time_index < valid_lengths[start:stop, None]
        if not np.array_equal(mask_chunk, expected_mask):
            raise Action0DatasetError(
                f"{valid_mask_path}: mask must be a contiguous right-padded prefix"
            )
        reconstruction_chunk = np.asarray(reconstructed[start:stop])
        if np.any(reconstruction_chunk[~mask_chunk] != 0.0):
            raise Action0DatasetError(
                f"{reconstructed_path}: reconstruction padding must be exact zero"
            )

    padding_summary = load_json_object(padding_summary_path)
    if int(padding_summary.get("target_length")) != padded_length:
        raise Action0DatasetError(
            f"{padding_summary_path}: target_length mismatch"
        )
    if padding_summary.get("padding_side") != "right":
        raise Action0DatasetError(f"{padding_summary_path}: expected padding_side='right'")
    if padding_summary.get("overflow_policy") != "skip":
        raise Action0DatasetError(f"{padding_summary_path}: expected overflow_policy='skip'")

    reconstruction_metadata = load_json_object(reconstruction_metadata_path)
    if reconstruction_metadata.get("artifact_type") != (
        "padded_segmentwise_custom_wavelet_acceleration_reconstruction"
    ):
        raise Action0DatasetError(
            f"{reconstruction_metadata_path}: unexpected artifact_type"
        )
    output_meta = reconstruction_metadata.get("output")
    if not isinstance(output_meta, dict):
        raise Action0DatasetError(
            f"{reconstruction_metadata_path}: missing output metadata"
        )
    if output_meta.get("shape") != [segment_count, padded_length, 3]:
        raise Action0DatasetError(
            f"{reconstruction_metadata_path}: output shape metadata mismatch"
        )
    if output_meta.get("units") != ["m/s^2", "m/s^2", "m/s^2"]:
        raise Action0DatasetError(
            f"{reconstruction_metadata_path}: expected m/s^2 output units"
        )

    manifest = pd.read_csv(padding_manifest_path, dtype=str, keep_default_na=False)
    required_columns = {
        "segment_index",
        "output_segment_index",
        "exported",
        "original_length",
        "target_length",
    }
    missing_columns = required_columns.difference(manifest.columns)
    if missing_columns:
        raise Action0DatasetError(
            f"{padding_manifest_path}: missing columns {sorted(missing_columns)}"
        )

    exported_mask = manifest["exported"].map(
        lambda value: parse_bool_value(value, context=str(padding_manifest_path))
    )
    exported = manifest.loc[exported_mask].copy()
    if len(exported) != segment_count:
        raise Action0DatasetError(
            f"{padding_manifest_path}: exported rows={len(exported)} "
            f"!= padded segments={segment_count}"
        )

    for field in ("segment_index", "output_segment_index", "original_length", "target_length"):
        exported[f"_{field}"] = [
            parse_manifest_int(value, field=field, path=padding_manifest_path)
            for value in exported[field]
        ]
    exported = exported.sort_values("_output_segment_index", kind="stable").reset_index(drop=True)
    if exported["_output_segment_index"].tolist() != list(range(segment_count)):
        raise Action0DatasetError(
            f"{padding_manifest_path}: output_segment_index must be contiguous 0..S-1"
        )
    if not np.array_equal(exported["_original_length"].to_numpy(np.int64), valid_lengths):
        raise Action0DatasetError(
            f"{padding_manifest_path}: original_length does not match valid_lengths"
        )
    if set(exported["_target_length"].tolist()) != {padded_length}:
        raise Action0DatasetError(
            f"{padding_manifest_path}: target_length inconsistent"
        )
    if "label" in exported.columns:
        manifest_labels = exported["label"].astype(str).to_numpy()
        if not np.array_equal(manifest_labels, labels):
            raise Action0DatasetError(
                f"{padding_manifest_path}: exported labels do not match padded labels"
            )

    provenance_rows: list[dict[str, object]] = []
    for output_index, row in exported.iterrows():
        provenance_rows.append(
            {
                "sample_id": f"{user}/action_{action}/{stem}/segment_{output_index:06d}",
                "user": user,
                "action": action,
                "stem": stem,
                "segment_index": int(output_index),
                "source_segment_index": int(row["_segment_index"]),
                "dataset_id": row.get("dataset_id", ""),
                "label": str(labels[output_index]),
                "valid_length": int(valid_lengths[output_index]),
                "package_relative_dir": str(action_dir.relative_to(padded_root)),
            }
        )

    package = PairedPackage(
        user=user,
        action=action,
        stem=stem,
        directory=action_dir,
        padded_spike_imu_path=padded_path,
        reconstructed_acceleration_path=reconstructed_path,
        reconstruction_metadata_path=reconstruction_metadata_path,
        labels_path=labels_path,
        valid_lengths_path=valid_lengths_path,
        valid_mask_path=valid_mask_path,
        padding_manifest_path=padding_manifest_path,
        padding_summary_path=padding_summary_path,
        padded_spike_imu=padded_spike,
        reconstructed_acceleration=reconstructed,
        labels=labels,
        valid_lengths=valid_lengths,
        valid_mask=valid_mask,
    )
    return package, pd.DataFrame(provenance_rows)


def load_all_paired_packages(
    padded_root: Path,
    *,
    root_target_length: int,
) -> tuple[tuple[PairedPackage, ...], pd.DataFrame]:
    packages: list[PairedPackage] = []
    manifest_parts: list[pd.DataFrame] = []
    seen_identity: set[tuple[str, str]] = set()

    for padded_path in discover_padded_spike_paths(padded_root):
        package, rows = load_and_validate_paired_package(
            padded_path,
            padded_root=padded_root,
            root_target_length=root_target_length,
        )
        identity = (package.user, package.action)
        if identity in seen_identity:
            raise Action0DatasetError(f"Duplicate package identity: {identity}")
        seen_identity.add(identity)
        rows["package_index"] = len(packages)
        packages.append(package)
        manifest_parts.append(rows)

    sample_manifest = pd.concat(manifest_parts, ignore_index=True)
    if sample_manifest["sample_id"].duplicated().any():
        raise Action0DatasetError("Duplicate sample IDs found")
    return tuple(packages), sample_manifest

In [ ]:
PADDED_ROOT = resolve_padded_root(ROOT)
producer_metadata = load_padding_dataset_metadata(PADDED_ROOT)

if producer_metadata.input_kind != "spike-imu":
    raise Action0DatasetError(
        f"Expected input_kind='spike-imu', got {producer_metadata.input_kind!r}"
    )
if producer_metadata.feature_schema != SPIKE_IMU_FEATURE_SCHEMA:
    raise Action0DatasetError(
        f"Expected feature_schema={SPIKE_IMU_FEATURE_SCHEMA!r}, "
        f"got {producer_metadata.feature_schema!r}"
    )
if producer_metadata.channel_count != SPIKE_IMU_CHANNEL_COUNT:
    raise Action0DatasetError(
        f"Expected {SPIKE_IMU_CHANNEL_COUNT} channels, got {producer_metadata.channel_count}"
    )
if producer_metadata.padding_side != "right":
    raise Action0DatasetError("This notebook requires canonical right padding")

packages, sample_manifest = load_all_paired_packages(
    PADDED_ROOT,
    root_target_length=producer_metadata.target_length,
)

print(f"Padded root:       {PADDED_ROOT}")
print(f"Packages:          {len(packages)}")
print(f"Users:             {sample_manifest['user'].nunique()}")
print(f"Actions:           {sample_manifest['action'].nunique()}")
print(f"Segments:          {len(sample_manifest)}")
print(f"Target length:     {producer_metadata.target_length}")
print(f"Sampling rate:     {producer_metadata.sampling_rate_hz:g} Hz")
display(sample_manifest.head())

## 4. Load the raw-trained baseline checkpoint and recover the exact experiment contract

Experiment B must inherit the baseline:

- model weights,
- class mapping,
- train / validation / test users,
- raw-train acceleration normalization.

Nothing is estimated from the reconstructed signals here.

In [ ]:
def load_checkpoint_compat(path: Path) -> dict[str, object]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Baseline checkpoint not found: {path}\n"
            "Run the raw acceleration CNN notebook with SAVE_ARTIFACTS=True first, "
            "or edit BASELINE_CHECKPOINT_PATH."
        )
    try:
        checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        checkpoint = torch.load(path, map_location="cpu")
    if not isinstance(checkpoint, dict):
        raise TypeError(f"Expected checkpoint dict, got {type(checkpoint).__name__}")
    return checkpoint


checkpoint = load_checkpoint_compat(BASELINE_CHECKPOINT_PATH)

required_checkpoint_keys = {
    "model_state_dict",
    "model_config",
    "class_to_idx",
    "normalization_mean",
    "normalization_std",
    "train_users",
    "val_users",
    "test_users",
}
missing_checkpoint_keys = sorted(required_checkpoint_keys.difference(checkpoint))
if missing_checkpoint_keys:
    raise KeyError(f"Checkpoint is missing keys: {missing_checkpoint_keys}")

class_to_idx = {str(k): int(v) for k, v in dict(checkpoint["class_to_idx"]).items()}
idx_to_class = {index: label for label, index in class_to_idx.items()}
if sorted(idx_to_class) != list(range(len(class_to_idx))):
    raise ValueError("Checkpoint class_to_idx values must be contiguous 0..C-1")

TRAIN_USERS = [str(value) for value in checkpoint["train_users"]]
VAL_USERS = [str(value) for value in checkpoint["val_users"]]
TEST_USERS = [str(value) for value in checkpoint["test_users"]]
_validate_user_splits(TRAIN_USERS, VAL_USERS, TEST_USERS)

acceleration_mean = np.asarray(checkpoint["normalization_mean"], dtype=np.float64)
acceleration_std = np.asarray(checkpoint["normalization_std"], dtype=np.float64)
if acceleration_mean.shape != (3,) or acceleration_std.shape != (3,):
    raise ValueError(
        f"Checkpoint normalization must be 3-D; got mean={acceleration_mean.shape}, "
        f"std={acceleration_std.shape}"
    )
if not np.isfinite(acceleration_mean).all() or not np.isfinite(acceleration_std).all():
    raise ValueError("Checkpoint normalization contains non-finite values")
if np.any(acceleration_std <= 0):
    raise ValueError("Checkpoint normalization std must be positive")

checkpoint_slice = checkpoint.get("acceleration_slice")
if checkpoint_slice is not None and list(checkpoint_slice) != [15, 18]:
    raise ValueError(
        f"Checkpoint acceleration_slice={checkpoint_slice}; this notebook expects [15,18]"
    )

available_users = set(sample_manifest["user"].unique())
assigned_users = set(TRAIN_USERS) | set(VAL_USERS) | set(TEST_USERS)
missing_users = sorted(assigned_users - available_users, key=natural_key)
unused_users = sorted(available_users - assigned_users, key=natural_key)
if missing_users:
    raise FileNotFoundError(f"Checkpoint users absent from PADDED_ROOT: {missing_users}")
if unused_users:
    raise ValueError(
        "Current dataset contains users not assigned by the baseline checkpoint: "
        f"{unused_users}. Use the exact same dataset root as Experiment A."
    )

dataset_labels = set(sample_manifest["label"].astype(str).unique())
checkpoint_labels = set(class_to_idx)
if dataset_labels != checkpoint_labels:
    raise ValueError(
        "Dataset label set differs from checkpoint class mapping.\n"
        f"Only in dataset: {sorted(dataset_labels - checkpoint_labels)}\n"
        f"Only in checkpoint: {sorted(checkpoint_labels - dataset_labels)}"
    )

split_by_user = {
    **{user: "train" for user in TRAIN_USERS},
    **{user: "val" for user in VAL_USERS},
    **{user: "test" for user in TEST_USERS},
}
sample_manifest["split"] = sample_manifest["user"].map(split_by_user)
if sample_manifest["split"].isna().any():
    raise AssertionError("Every sample must inherit a split from the checkpoint")
sample_manifest["label_idx"] = (
    sample_manifest["label"].map(class_to_idx).astype(np.int64)
)

split_summary = (
    sample_manifest.groupby("split")
    .agg(users=("user", "nunique"), samples=("sample_id", "size"), labels=("label", "nunique"))
    .reindex(["train", "val", "test"])
)

print(f"Train users ({len(TRAIN_USERS)}): {TRAIN_USERS}")
print(f"Val users   ({len(VAL_USERS)}): {VAL_USERS}")
print(f"Test users  ({len(TEST_USERS)}): {TEST_USERS}")
print(f"Baseline best epoch: {checkpoint.get('best_epoch', 'unknown')}")
print(
    "Baseline best validation BA:",
    checkpoint.get("best_val_balanced_accuracy", "unknown"),
)
display(split_summary)
display(
    pd.DataFrame(
        {
            "channel": ACCELERATION_CHANNEL_NAMES,
            "raw_train_mean_m_s2": acceleration_mean,
            "raw_train_std_m_s2": acceleration_std,
        }
    )
)

## 5. Raw / reconstructed paired datasets

Both modes use the **same baseline raw-train mean/std**.

- `mode="raw"` reads channels `15:18` from `paddedSpikeIMU`.
- `mode="recon"` reads the standalone reconstructed `(T,3)` acceleration.

Padding is set to exact zero **after** normalization, matching the baseline CNN contract.

In [ ]:
class PairedAccelerationDataset(Dataset[dict[str, object]]):
    def __init__(
        self,
        packages: Sequence[PairedPackage],
        sample_manifest: pd.DataFrame,
        *,
        split: str,
        mode: str,
        mean: np.ndarray,
        std: np.ndarray,
    ) -> None:
        if mode not in {"raw", "recon"}:
            raise ValueError("mode must be 'raw' or 'recon'")
        rows = (
            sample_manifest.loc[sample_manifest["split"] == split]
            .copy()
            .reset_index(drop=True)
        )
        if rows.empty:
            raise ValueError(f"Split {split!r} contains no samples")

        self.packages = tuple(packages)
        self.split = split
        self.mode = mode
        self.package_indices = rows["package_index"].to_numpy(np.int64)
        self.segment_indices = rows["segment_index"].to_numpy(np.int64)
        self.labels = rows["label_idx"].to_numpy(np.int64)
        self.valid_lengths = rows["valid_length"].to_numpy(np.int64)
        self.sample_ids = rows["sample_id"].astype(str).tolist()
        self.mean = np.asarray(mean, dtype=np.float32).reshape(1, 3)
        self.std = np.asarray(std, dtype=np.float32).reshape(1, 3)
        self.padded_length = int(self.packages[int(self.package_indices[0])].padded_length)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int) -> dict[str, object]:
        package = self.packages[int(self.package_indices[index])]
        segment_index = int(self.segment_indices[index])

        if self.mode == "raw":
            acceleration = np.array(
                package.padded_spike_imu[segment_index, :, ACCELERATION_SLICE],
                dtype=np.float32,
                copy=True,
            )
        else:
            acceleration = np.array(
                package.reconstructed_acceleration[segment_index],
                dtype=np.float32,
                copy=True,
            )

        mask = np.array(
            package.valid_mask[segment_index],
            dtype=np.bool_,
            copy=True,
        )
        valid_length = int(self.valid_lengths[index])

        if acceleration.shape != (self.padded_length, 3):
            raise AssertionError(f"Unexpected acceleration shape: {acceleration.shape}")
        if mask.shape != (self.padded_length,):
            raise AssertionError(f"Unexpected mask shape: {mask.shape}")
        if int(mask.sum()) != valid_length:
            raise AssertionError("valid_mask.sum() must equal valid_length")

        acceleration = (acceleration - self.mean) / self.std
        acceleration[~mask] = 0.0

        if not np.isfinite(acceleration).all():
            raise FloatingPointError(
                f"Non-finite normalized {self.mode} sample: {self.sample_ids[index]}"
            )
        if np.any(acceleration[~mask] != 0.0):
            raise AssertionError("Normalized padding must be exact zero")

        return {
            "x": torch.from_numpy(np.ascontiguousarray(acceleration.T)),
            "label": torch.tensor(int(self.labels[index]), dtype=torch.long),
            "valid_mask": torch.from_numpy(mask),
            "valid_length": torch.tensor(valid_length, dtype=torch.long),
            "sample_id": self.sample_ids[index],
        }


def make_loader(
    dataset: Dataset[dict[str, object]],
    *,
    shuffle: bool = False,
    seed: int = RANDOM_SEED,
) -> DataLoader[dict[str, object]]:
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        generator=generator if shuffle else None,
        num_workers=NUM_WORKERS,
        pin_memory=bool(USE_GPU and torch.cuda.is_available()),
        persistent_workers=NUM_WORKERS > 0,
    )


datasets = {}
loaders = {}
for split in ("train", "val", "test"):
    raw_dataset = PairedAccelerationDataset(
        packages,
        sample_manifest,
        split=split,
        mode="raw",
        mean=acceleration_mean,
        std=acceleration_std,
    )
    datasets[("raw", split)] = raw_dataset
    loaders[("raw", split)] = make_loader(raw_dataset)

# Experiment B needs reconstruction only for the held-out test users.
recon_test_dataset = PairedAccelerationDataset(
    packages,
    sample_manifest,
    split="test",
    mode="recon",
    mean=acceleration_mean,
    std=acceleration_std,
)
datasets[("recon", "test")] = recon_test_dataset
loaders[("recon", "test")] = make_loader(recon_test_dataset)

raw_test_ids = datasets[("raw", "test")].sample_ids
recon_test_ids = datasets[("recon", "test")].sample_ids
if raw_test_ids != recon_test_ids:
    raise AssertionError("Raw and reconstruction test sample order must be identical")

first_raw = next(iter(loaders[("raw", "test")]))
first_recon = next(iter(loaders[("recon", "test")]))
print("Raw test batch x:", tuple(first_raw["x"].shape))
print("Recon test batch x:", tuple(first_recon["x"].shape))
print("Paired IDs identical:", first_raw["sample_id"] == first_recon["sample_id"])

## 6. Recreate the baseline CNN architecture and load frozen weights

In [ ]:
def conv1d_output_lengths(
    lengths: torch.Tensor,
    *,
    kernel_size: int,
    stride: int = 1,
    padding: int = 0,
    dilation: int = 1,
) -> torch.Tensor:
    numerator = lengths + 2 * padding - dilation * (kernel_size - 1) - 1
    return torch.div(numerator, stride, rounding_mode="floor") + 1


def prefix_mask(lengths: torch.Tensor, time_steps: int) -> torch.Tensor:
    if lengths.ndim != 1:
        raise ValueError(f"lengths must have shape (B,), got {tuple(lengths.shape)}")
    if torch.any(lengths <= 0) or torch.any(lengths > time_steps):
        raise ValueError("Invalid propagated valid lengths")
    return (
        torch.arange(time_steps, device=lengths.device).unsqueeze(0)
        < lengths.unsqueeze(1)
    )


class MaskAwareAccelerationCNN(nn.Module):
    def __init__(self, num_classes: int) -> None:
        super().__init__()
        if num_classes <= 0:
            raise ValueError("num_classes must be positive")
        self.num_classes = int(num_classes)
        self.embedding_dim = 256

        self.conv1 = nn.Conv1d(3, 64, kernel_size=7, stride=1, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(64)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=5, stride=2, padding=2, bias=False)
        self.bn2 = nn.BatchNorm1d(128)
        self.conv3 = nn.Conv1d(128, 256, kernel_size=5, stride=2, padding=2, bias=False)
        self.bn3 = nn.BatchNorm1d(256)
        self.conv4 = nn.Conv1d(256, 256, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn4 = nn.BatchNorm1d(256)
        self.classifier = nn.Linear(self.embedding_dim, self.num_classes)

    @staticmethod
    def _masked_feature(feature: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        return feature * mask.unsqueeze(1).to(dtype=feature.dtype)

    def forward(
        self,
        x: torch.Tensor,
        *,
        valid_mask: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        if x.ndim != 3 or x.shape[1] != 3:
            raise ValueError(f"x must have shape (B,3,T), got {tuple(x.shape)}")
        if valid_mask.shape != (x.shape[0], x.shape[2]):
            raise ValueError("valid_mask shape does not match x")

        valid_mask = valid_mask.to(device=x.device, dtype=torch.bool)
        if not valid_mask.any(dim=1).all():
            raise ValueError("Every segment must contain at least one valid time step")

        lengths = valid_mask.sum(dim=1).long()
        x = x * valid_mask.unsqueeze(1).to(dtype=x.dtype)

        x = F.relu(self.bn1(self.conv1(x)))
        lengths = conv1d_output_lengths(lengths, kernel_size=7, stride=1, padding=3)
        mask = prefix_mask(lengths, x.shape[-1])
        x = self._masked_feature(x, mask)

        x = F.relu(self.bn2(self.conv2(x)))
        lengths = conv1d_output_lengths(lengths, kernel_size=5, stride=2, padding=2)
        mask = prefix_mask(lengths, x.shape[-1])
        x = self._masked_feature(x, mask)

        x = F.relu(self.bn3(self.conv3(x)))
        lengths = conv1d_output_lengths(lengths, kernel_size=5, stride=2, padding=2)
        mask = prefix_mask(lengths, x.shape[-1])
        x = self._masked_feature(x, mask)

        x = F.relu(self.bn4(self.conv4(x)))
        lengths = conv1d_output_lengths(lengths, kernel_size=3, stride=1, padding=1)
        mask = prefix_mask(lengths, x.shape[-1])
        x = self._masked_feature(x, mask)

        weights = mask.unsqueeze(1).to(dtype=x.dtype)
        embedding = (x * weights).sum(dim=-1) / weights.sum(dim=-1).clamp_min(1.0)
        logits = self.classifier(embedding)
        return logits, embedding

    def architecture_config(self) -> dict[str, object]:
        return {
            "input_channels": 3,
            "blocks": [
                {"out_channels": 64, "kernel_size": 7, "stride": 1, "padding": 3},
                {"out_channels": 128, "kernel_size": 5, "stride": 2, "padding": 2},
                {"out_channels": 256, "kernel_size": 5, "stride": 2, "padding": 2},
                {"out_channels": 256, "kernel_size": 3, "stride": 1, "padding": 1},
            ],
            "normalization": "BatchNorm1d",
            "activation": "ReLU",
            "pooling": "valid-length-propagated masked global average pooling",
            "embedding_dim": self.embedding_dim,
            "num_classes": self.num_classes,
        }


_set_random_seed(RANDOM_SEED)
DEVICE = _resolve_device(USE_GPU)
model = MaskAwareAccelerationCNN(num_classes=len(class_to_idx)).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.eval()

for parameter in model.parameters():
    parameter.requires_grad_(False)

checkpoint_model_config = checkpoint.get("model_config")
current_model_config = model.architecture_config()
if checkpoint_model_config is not None and checkpoint_model_config != current_model_config:
    raise ValueError(
        "Current CNN architecture does not exactly match the checkpoint model_config.\n"
        f"Checkpoint: {checkpoint_model_config}\n"
        f"Current:    {current_model_config}"
    )

print(f"Device: {DEVICE}")
print("CNN checkpoint loaded and frozen.")
print(f"Trainable CNN parameters: {sum(p.requires_grad for p in model.parameters())}")

## 7. Direct Experiment B classification

This is the most direct downstream test:

- `raw test → frozen raw CNN` = reference,
- `reconstructed test → the exact same frozen raw CNN` = Experiment B.

A large drop can reflect lost task information **and/or** a raw→reconstruction domain shift. Later cells help diagnose which representation properties changed.

In [ ]:
@torch.inference_mode()
def evaluate_cnn(
    model: nn.Module,
    loader: DataLoader[dict[str, object]],
    *,
    device: torch.device,
) -> dict[str, object]:
    model.eval()
    criterion = nn.CrossEntropyLoss(reduction="sum")
    loss_sum = 0.0
    y_true: list[int] = []
    y_pred: list[int] = []
    logits_parts: list[np.ndarray] = []
    sample_ids: list[str] = []

    for batch in loader:
        x = batch["x"].to(device=device, dtype=torch.float32, non_blocking=True)
        labels = batch["label"].to(device=device, dtype=torch.long, non_blocking=True)
        valid_mask = batch["valid_mask"].to(
            device=device,
            dtype=torch.bool,
            non_blocking=True,
        )
        logits, _ = model(x, valid_mask=valid_mask)
        loss_sum += float(criterion(logits, labels).detach())
        prediction = logits.argmax(dim=1)

        y_true.extend(labels.cpu().tolist())
        y_pred.extend(prediction.cpu().tolist())
        logits_parts.append(logits.cpu().numpy().astype(np.float32, copy=False))
        sample_ids.extend(str(v) for v in batch["sample_id"])

    metrics = _classification_metrics(y_true, y_pred)
    metrics["loss"] = loss_sum / len(y_true)
    return {
        "metrics": metrics,
        "y": np.asarray(y_true, dtype=np.int64),
        "pred": np.asarray(y_pred, dtype=np.int64),
        "logits": np.concatenate(logits_parts, axis=0),
        "sample_id": np.asarray(sample_ids, dtype=str),
    }


raw_cnn_eval = evaluate_cnn(model, loaders[("raw", "test")], device=DEVICE)
recon_cnn_eval = evaluate_cnn(model, loaders[("recon", "test")], device=DEVICE)

if not np.array_equal(raw_cnn_eval["sample_id"], recon_cnn_eval["sample_id"]):
    raise AssertionError("Raw/reconstruction test sample IDs do not align")
if not np.array_equal(raw_cnn_eval["y"], recon_cnn_eval["y"]):
    raise AssertionError("Raw/reconstruction labels do not align")

classification_comparison = pd.DataFrame(
    {
        "raw_test": raw_cnn_eval["metrics"],
        "recon_test_B": recon_cnn_eval["metrics"],
    }
)
classification_comparison["delta_recon_minus_raw"] = (
    classification_comparison["recon_test_B"] - classification_comparison["raw_test"]
)
display(classification_comparison)

In [ ]:
def confusion_matrix_numpy(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    num_classes: int,
) -> np.ndarray:
    matrix = np.zeros((num_classes, num_classes), dtype=np.int64)
    np.add.at(matrix, (y_true.astype(np.int64), y_pred.astype(np.int64)), 1)
    return matrix


recon_cm = confusion_matrix_numpy(
    recon_cnn_eval["y"],
    recon_cnn_eval["pred"],
    len(class_to_idx),
)

plt.figure(figsize=(max(8, len(class_to_idx) * 0.28), max(7, len(class_to_idx) * 0.25)))
plt.imshow(recon_cm, aspect="auto")
plt.colorbar(label="Count")
labels_in_index_order = [idx_to_class[i] for i in range(len(class_to_idx))]
plt.xticks(np.arange(len(class_to_idx)), labels_in_index_order, rotation=90)
plt.yticks(np.arange(len(class_to_idx)), labels_in_index_order)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Experiment B — reconstructed test → frozen raw-trained CNN")
plt.tight_layout()
if SAVE_FIGURES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    plt.savefig(OUTPUT_DIR / "recon_test_confusion_matrix.png", dpi=180)
plt.show()

## 8. Extract raw and reconstructed 256-D embeddings

We extract:

- **raw train**: fixed gallery / prototype / probe training features,
- **raw validation**: kNN-K selection and probe validation,
- **raw test**: paired reference,
- **reconstructed test**: Experiment B query features.

`z` is the L2-normalized 256-D embedding used for cosine geometry.

In [ ]:
@torch.inference_mode()
def extract_embeddings(
    model: nn.Module,
    loader: DataLoader[dict[str, object]],
    *,
    device: torch.device,
) -> dict[str, np.ndarray]:
    model.eval()
    h_parts: list[np.ndarray] = []
    y_parts: list[np.ndarray] = []
    pred_parts: list[np.ndarray] = []
    logit_parts: list[np.ndarray] = []
    sample_ids: list[str] = []

    for batch in loader:
        x = batch["x"].to(device=device, dtype=torch.float32, non_blocking=True)
        valid_mask = batch["valid_mask"].to(
            device=device,
            dtype=torch.bool,
            non_blocking=True,
        )
        labels = batch["label"].long()
        logits, h = model(x, valid_mask=valid_mask)

        h_parts.append(h.cpu().numpy().astype(np.float32, copy=False))
        y_parts.append(labels.numpy().astype(np.int64, copy=False))
        pred_parts.append(
            logits.argmax(dim=1).cpu().numpy().astype(np.int64, copy=False)
        )
        logit_parts.append(logits.cpu().numpy().astype(np.float32, copy=False))
        sample_ids.extend(str(value) for value in batch["sample_id"])

    h_all = np.concatenate(h_parts, axis=0)
    y_all = np.concatenate(y_parts, axis=0)
    pred_all = np.concatenate(pred_parts, axis=0)
    logits_all = np.concatenate(logit_parts, axis=0)

    norms = np.linalg.norm(h_all, axis=1, keepdims=True)
    if np.any(norms <= 0) or not np.isfinite(norms).all():
        raise FloatingPointError("Every embedding must have finite positive L2 norm")
    z_all = h_all / norms

    return {
        "h": h_all,
        "z": z_all.astype(np.float32, copy=False),
        "y": y_all,
        "cnn_pred": pred_all,
        "logits": logits_all,
        "sample_id": np.asarray(sample_ids, dtype=str),
    }


raw_train = extract_embeddings(model, loaders[("raw", "train")], device=DEVICE)
raw_val = extract_embeddings(model, loaders[("raw", "val")], device=DEVICE)
raw_test = extract_embeddings(model, loaders[("raw", "test")], device=DEVICE)
recon_test = extract_embeddings(model, loaders[("recon", "test")], device=DEVICE)

if not np.array_equal(raw_test["sample_id"], recon_test["sample_id"]):
    raise AssertionError("Paired test embedding IDs do not align")
if not np.array_equal(raw_test["y"], recon_test["y"]):
    raise AssertionError("Paired test embedding labels do not align")
if not np.array_equal(raw_test["cnn_pred"], raw_cnn_eval["pred"]):
    raise AssertionError("Raw embedding pass and raw classification pass disagree")
if not np.array_equal(recon_test["cnn_pred"], recon_cnn_eval["pred"]):
    raise AssertionError("Recon embedding pass and recon classification pass disagree")

for name, bundle in (
    ("raw_train", raw_train),
    ("raw_val", raw_val),
    ("raw_test", raw_test),
    ("recon_test", recon_test),
):
    print(name, "h=", bundle["h"].shape, "z=", bundle["z"].shape)

## 9. Paired preservation diagnostics

Because raw and reconstructed samples are one-to-one aligned, we can directly measure how far reconstruction moves each sample in the frozen CNN representation space.

- `embedding_cosine_similarity` close to 1: frozen-CNN semantics are well preserved.
- `embedding_cosine_distance = 1 - similarity`: lower is more preserved.
- `prediction_agreement`: how often the frozen CNN makes the same decision before/after reconstruction.
- transition table distinguishes `raw correct → recon wrong` from `raw wrong → recon correct`.

In [ ]:
paired_cosine_similarity = np.sum(
    np.asarray(raw_test["z"], dtype=np.float64)
    * np.asarray(recon_test["z"], dtype=np.float64),
    axis=1,
)
paired_cosine_similarity = np.clip(paired_cosine_similarity, -1.0, 1.0)
paired_cosine_distance = 1.0 - paired_cosine_similarity

raw_correct = raw_test["cnn_pred"] == raw_test["y"]
recon_correct = recon_test["cnn_pred"] == recon_test["y"]
prediction_agreement = float(
    np.mean(raw_test["cnn_pred"] == recon_test["cnn_pred"])
)

transition_labels = np.select(
    [
        raw_correct & recon_correct,
        raw_correct & ~recon_correct,
        ~raw_correct & recon_correct,
        ~raw_correct & ~recon_correct,
    ],
    [
        "raw_correct -> recon_correct",
        "raw_correct -> recon_wrong",
        "raw_wrong -> recon_correct",
        "raw_wrong -> recon_wrong",
    ],
    default="unexpected",
)

transition_table = (
    pd.Series(transition_labels, name="transition")
    .value_counts()
    .rename_axis("transition")
    .reset_index(name="count")
)
transition_table["fraction"] = transition_table["count"] / len(raw_test["y"])

paired_summary = pd.DataFrame(
    [
        {
            "mean_embedding_cosine_similarity": float(paired_cosine_similarity.mean()),
            "median_embedding_cosine_similarity": float(np.median(paired_cosine_similarity)),
            "mean_embedding_cosine_distance": float(paired_cosine_distance.mean()),
            "median_embedding_cosine_distance": float(np.median(paired_cosine_distance)),
            "prediction_agreement": prediction_agreement,
        }
    ]
)

paired_per_class = []
for class_index in np.unique(raw_test["y"]):
    selected = raw_test["y"] == class_index
    paired_per_class.append(
        {
            "label_idx": int(class_index),
            "label": idx_to_class[int(class_index)],
            "samples": int(selected.sum()),
            "mean_embedding_cosine_similarity": float(
                paired_cosine_similarity[selected].mean()
            ),
            "mean_embedding_cosine_distance": float(
                paired_cosine_distance[selected].mean()
            ),
            "prediction_agreement": float(
                np.mean(
                    raw_test["cnn_pred"][selected]
                    == recon_test["cnn_pred"][selected]
                )
            ),
        }
    )
paired_per_class = pd.DataFrame(paired_per_class)

display(paired_summary)
display(transition_table)
display(paired_per_class)

## 10. Retrieval and kNN: reconstruction queries against the unchanged raw-train gallery

This is deliberately asymmetric:

```text
query  = reconstructed held-out test
gallery = raw training users
```

That matches Experiment B's "do not adapt the downstream system to reconstruction" principle.

The kNN `K` is selected only from **raw validation → raw train**, then frozen for both raw-test and reconstructed-test evaluation.

In [ ]:
def macro_average_by_class(values: np.ndarray, labels: np.ndarray) -> float:
    per_class = [
        float(values[labels == class_index].mean())
        for class_index in np.unique(labels)
    ]
    return float(np.mean(per_class))


def cosine_topk(
    query_z: np.ndarray,
    gallery_z: np.ndarray,
    *,
    max_k: int,
    batch_size: int = 128,
) -> tuple[np.ndarray, np.ndarray]:
    query = np.asarray(query_z, dtype=np.float32)
    gallery = np.asarray(gallery_z, dtype=np.float32)
    if max_k <= 0 or max_k > len(gallery):
        raise ValueError(f"max_k must be in [1,{len(gallery)}]")

    all_indices = []
    all_scores = []
    for start in range(0, len(query), batch_size):
        stop = min(start + batch_size, len(query))
        scores = query[start:stop] @ gallery.T
        candidate = np.argpartition(-scores, kth=max_k - 1, axis=1)[:, :max_k]
        candidate_scores = np.take_along_axis(scores, candidate, axis=1)
        order = np.argsort(-candidate_scores, axis=1, kind="stable")
        all_indices.append(np.take_along_axis(candidate, order, axis=1).astype(np.int64))
        all_scores.append(
            np.take_along_axis(candidate_scores, order, axis=1).astype(np.float32)
        )
    return np.concatenate(all_indices), np.concatenate(all_scores)


def knn_predict_from_topk(
    neighbour_indices: np.ndarray,
    neighbour_scores: np.ndarray,
    gallery_y: np.ndarray,
    *,
    k: int,
) -> np.ndarray:
    predictions = np.empty(len(neighbour_indices), dtype=np.int64)
    for query_index in range(len(neighbour_indices)):
        labels = gallery_y[neighbour_indices[query_index, :k]]
        scores = neighbour_scores[query_index, :k]
        unique, counts = np.unique(labels, return_counts=True)
        max_count = counts.max()
        candidates = unique[counts == max_count]
        if len(candidates) == 1:
            predictions[query_index] = int(candidates[0])
            continue
        score_sums = {
            int(label): float(scores[labels == label].sum())
            for label in candidates
        }
        best_score = max(score_sums.values())
        predictions[query_index] = min(
            label for label, value in score_sums.items() if value == best_score
        )
    return predictions


def same_label_at_k(
    neighbour_indices: np.ndarray,
    *,
    query_y: np.ndarray,
    gallery_y: np.ndarray,
    k: int,
) -> tuple[float, float]:
    relevant = gallery_y[neighbour_indices[:, :k]] == query_y[:, None]
    per_query = relevant.mean(axis=1, dtype=np.float64)
    return float(per_query.mean()), macro_average_by_class(per_query, query_y)


def retrieval_map(
    query_z: np.ndarray,
    gallery_z: np.ndarray,
    *,
    query_y: np.ndarray,
    gallery_y: np.ndarray,
    batch_size: int = 128,
) -> tuple[float, float, np.ndarray]:
    aps = np.empty(len(query_z), dtype=np.float64)
    ranks = np.arange(1, len(gallery_z) + 1, dtype=np.float64)

    for start in range(0, len(query_z), batch_size):
        stop = min(start + batch_size, len(query_z))
        scores = (
            np.asarray(query_z[start:stop], dtype=np.float32)
            @ np.asarray(gallery_z, dtype=np.float32).T
        )
        ordering = np.argsort(-scores, axis=1, kind="stable")

        for local_index, global_index in enumerate(range(start, stop)):
            relevant_count = int(
                np.count_nonzero(gallery_y == query_y[global_index])
            )
            if relevant_count <= 0:
                raise ValueError(
                    f"Query {global_index} has no same-label item in raw train gallery"
                )
            relevance = gallery_y[ordering[local_index]] == query_y[global_index]
            precision = np.cumsum(relevance, dtype=np.float64) / ranks
            aps[global_index] = float(
                (precision * relevance).sum() / relevant_count
            )

    return float(aps.mean()), macro_average_by_class(aps, query_y), aps


def select_knn_k(
    *,
    val_z: np.ndarray,
    val_y: np.ndarray,
    train_z: np.ndarray,
    train_y: np.ndarray,
    candidates: Sequence[int],
) -> tuple[int, pd.DataFrame]:
    candidates = sorted(set(int(value) for value in candidates))
    indices, scores = cosine_topk(
        val_z,
        train_z,
        max_k=max(candidates),
        batch_size=SIMILARITY_QUERY_BATCH_SIZE,
    )
    rows = []
    for k in candidates:
        prediction = knn_predict_from_topk(indices, scores, train_y, k=k)
        rows.append(
            {
                "k": k,
                **_classification_metrics(
                    val_y.tolist(),
                    prediction.tolist(),
                ),
            }
        )
    table = pd.DataFrame(rows).sort_values("k").reset_index(drop=True)
    best = table.sort_values(
        ["balanced_accuracy", "k"],
        ascending=[False, True],
        kind="stable",
    ).iloc[0]
    return int(best["k"]), table


max_requested_k = max(*K_VALUES, *KNN_K_CANDIDATES)
if max_requested_k > len(raw_train["y"]):
    raise ValueError(
        f"Largest K={max_requested_k} exceeds raw train gallery size={len(raw_train['y'])}"
    )

selected_knn_k, validation_knn_table = select_knn_k(
    val_z=raw_val["z"],
    val_y=raw_val["y"],
    train_z=raw_train["z"],
    train_y=raw_train["y"],
    candidates=KNN_K_CANDIDATES,
)
print(f"Selected K from raw validation → raw train: {selected_knn_k}")
display(validation_knn_table)


def evaluate_retrieval_bundle(
    query_bundle: dict[str, np.ndarray],
    *,
    name: str,
) -> tuple[dict[str, float], pd.DataFrame, np.ndarray]:
    topk_indices, topk_scores = cosine_topk(
        query_bundle["z"],
        raw_train["z"],
        max_k=max(max(K_VALUES), selected_knn_k),
        batch_size=SIMILARITY_QUERY_BATCH_SIZE,
    )
    knn_prediction = knn_predict_from_topk(
        topk_indices,
        topk_scores,
        raw_train["y"],
        k=selected_knn_k,
    )
    knn_metrics = _classification_metrics(
        query_bundle["y"].tolist(),
        knn_prediction.tolist(),
    )

    same_rows = []
    for k in K_VALUES:
        micro, macro = same_label_at_k(
            topk_indices,
            query_y=query_bundle["y"],
            gallery_y=raw_train["y"],
            k=k,
        )
        same_rows.append(
            {
                "source": name,
                "k": k,
                "same_label_micro": micro,
                "same_label_macro": macro,
            }
        )

    map_micro, map_macro, aps = retrieval_map(
        query_bundle["z"],
        raw_train["z"],
        query_y=query_bundle["y"],
        gallery_y=raw_train["y"],
        batch_size=SIMILARITY_QUERY_BATCH_SIZE,
    )
    summary = {
        "kNN_balanced_accuracy": knn_metrics["balanced_accuracy"],
        "kNN_accuracy": knn_metrics["accuracy"],
        "kNN_macro_f1": knn_metrics["macro_f1"],
        "retrieval_mAP_micro": map_micro,
        "retrieval_mAP_macro": map_macro,
    }
    return summary, pd.DataFrame(same_rows), aps


raw_retrieval, raw_same_label, raw_ap = evaluate_retrieval_bundle(
    raw_test,
    name="raw_test",
)
recon_retrieval, recon_same_label, recon_ap = evaluate_retrieval_bundle(
    recon_test,
    name="recon_test_B",
)

retrieval_comparison = pd.DataFrame(
    {
        "raw_test": raw_retrieval,
        "recon_test_B": recon_retrieval,
    }
)
retrieval_comparison["delta_recon_minus_raw"] = (
    retrieval_comparison["recon_test_B"] - retrieval_comparison["raw_test"]
)
same_label_comparison = pd.concat(
    [raw_same_label, recon_same_label],
    ignore_index=True,
)

display(retrieval_comparison)
display(same_label_comparison)

In [ ]:
for metric in ("same_label_macro", "same_label_micro"):
    plt.figure(figsize=(8, 5))
    for source, table in same_label_comparison.groupby("source", sort=False):
        plt.plot(table["k"], table[metric], marker="o", label=source)
    plt.xlabel("K")
    plt.ylabel(metric)
    plt.title(f"Raw vs reconstruction — {metric}@K against raw-train gallery")
    plt.legend()
    plt.tight_layout()
    if SAVE_FIGURES:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        plt.savefig(OUTPUT_DIR / f"{metric}_at_k_raw_vs_recon.png", dpi=180)
    plt.show()

## 11. Test-only class geometry and raw-train prototype classification

Geometry is computed separately in `raw_test` and `recon_test_B`.

For the prototype diagnostic, class centroids are built **only from raw train embeddings** and then applied unchanged to both test inputs.

In [ ]:
def class_centroids(
    z: np.ndarray,
    y: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    classes = np.unique(y)
    centroids = []
    for class_index in classes:
        centroid = z[y == class_index].mean(axis=0, dtype=np.float64)
        norm = float(np.linalg.norm(centroid))
        if not math.isfinite(norm) or norm <= 0:
            raise FloatingPointError(f"Invalid centroid for class {class_index}")
        centroids.append((centroid / norm).astype(np.float32))
    return classes.astype(np.int64), np.stack(centroids)


def macro_intra_cosine_distance(
    z: np.ndarray,
    y: np.ndarray,
) -> tuple[float, pd.DataFrame]:
    rows = []
    for class_index in np.unique(y):
        class_z = np.asarray(z[y == class_index], dtype=np.float64)
        n = len(class_z)
        if n < 2:
            continue
        vector_sum = class_z.sum(axis=0)
        mean_pair_cosine = (float(vector_sum @ vector_sum) - n) / (n * (n - 1))
        rows.append(
            {
                "label_idx": int(class_index),
                "label": idx_to_class[int(class_index)],
                "samples": n,
                "intra_cosine_distance": float(1.0 - mean_pair_cosine),
            }
        )
    if not rows:
        raise ValueError("No class has at least two samples")
    table = pd.DataFrame(rows).sort_values("label_idx").reset_index(drop=True)
    return float(table["intra_cosine_distance"].mean()), table


def centroid_inter_cosine_distance(z: np.ndarray, y: np.ndarray) -> float:
    _, centroids = class_centroids(z, y)
    if len(centroids) < 2:
        raise ValueError("D_inter requires at least two classes")
    distances = 1.0 - centroids @ centroids.T
    upper = distances[np.triu_indices(len(centroids), k=1)]
    return float(upper.mean())


def prototype_classification(
    *,
    train_z: np.ndarray,
    train_y: np.ndarray,
    test_z: np.ndarray,
    test_y: np.ndarray,
) -> tuple[dict[str, float], np.ndarray]:
    classes, centroids = class_centroids(train_z, train_y)
    scores = np.asarray(test_z, dtype=np.float32) @ centroids.T
    prediction = classes[scores.argmax(axis=1)]
    return (
        _classification_metrics(test_y.tolist(), prediction.tolist()),
        prediction,
    )


def evaluate_geometry(
    bundle: dict[str, np.ndarray],
    *,
    source: str,
) -> tuple[dict[str, float], pd.DataFrame, np.ndarray]:
    d_intra, per_class = macro_intra_cosine_distance(
        bundle["z"],
        bundle["y"],
    )
    d_inter = centroid_inter_cosine_distance(
        bundle["z"],
        bundle["y"],
    )
    prototype_metrics, prototype_prediction = prototype_classification(
        train_z=raw_train["z"],
        train_y=raw_train["y"],
        test_z=bundle["z"],
        test_y=bundle["y"],
    )
    per_class = per_class.copy()
    per_class["source"] = source
    summary = {
        "D_intra_macro": d_intra,
        "D_inter_centroids": d_inter,
        "D_inter_over_D_intra": d_inter / (d_intra + 1e-12),
        "prototype_balanced_accuracy": prototype_metrics["balanced_accuracy"],
        "prototype_accuracy": prototype_metrics["accuracy"],
        "prototype_macro_f1": prototype_metrics["macro_f1"],
    }
    return summary, per_class, prototype_prediction


raw_geometry, raw_intra_by_class, raw_prototype_pred = evaluate_geometry(
    raw_test,
    source="raw_test",
)
recon_geometry, recon_intra_by_class, recon_prototype_pred = evaluate_geometry(
    recon_test,
    source="recon_test_B",
)

geometry_comparison = pd.DataFrame(
    {
        "raw_test": raw_geometry,
        "recon_test_B": recon_geometry,
    }
)
geometry_comparison["delta_recon_minus_raw"] = (
    geometry_comparison["recon_test_B"] - geometry_comparison["raw_test"]
)
intra_by_class = pd.concat(
    [raw_intra_by_class, recon_intra_by_class],
    ignore_index=True,
)

display(geometry_comparison)
display(intra_by_class)

## 12. Frozen-encoder linear probe diagnostic

The probe is trained **only on raw-train `h`**, selected on **raw-validation `h`**, and then evaluated on:

- raw test `h`,
- reconstructed test `h`.

So the probe itself is not adapted to reconstruction.

In [ ]:
def standardize_feature_splits(
    train_x: np.ndarray,
    val_x: np.ndarray,
    test_x: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    mean = train_x.mean(axis=0, dtype=np.float64)
    std = train_x.std(axis=0, dtype=np.float64)
    std = np.where(std < 1e-8, 1.0, std)
    transform = lambda values: ((values - mean) / std).astype(np.float32)
    return transform(train_x), transform(val_x), transform(test_x), mean, std


def apply_feature_standardization(
    values: np.ndarray,
    mean: np.ndarray,
    std: np.ndarray,
) -> np.ndarray:
    return ((values - mean) / std).astype(np.float32)


@torch.inference_mode()
def evaluate_linear_probe(
    probe: nn.Module,
    features: np.ndarray,
    labels: np.ndarray,
    *,
    device: torch.device,
) -> dict[str, float]:
    probe.eval()
    logits_parts = []
    for start in range(0, len(features), LINEAR_PROBE_BATCH_SIZE):
        stop = min(start + LINEAR_PROBE_BATCH_SIZE, len(features))
        batch = torch.from_numpy(features[start:stop]).to(
            device=device,
            dtype=torch.float32,
        )
        logits_parts.append(probe(batch).cpu().numpy())
    logits = np.concatenate(logits_parts)
    prediction = logits.argmax(axis=1).astype(np.int64)
    return _classification_metrics(labels.tolist(), prediction.tolist())


def train_linear_probe(
    train_x: np.ndarray,
    train_y: np.ndarray,
    val_x: np.ndarray,
    val_y: np.ndarray,
    *,
    num_classes: int,
    device: torch.device,
) -> tuple[nn.Module, pd.DataFrame, int, float]:
    _set_random_seed(RANDOM_SEED + 1)
    probe = nn.Linear(train_x.shape[1], num_classes).to(device)
    optimizer = torch.optim.Adam(
        probe.parameters(),
        lr=LINEAR_PROBE_LEARNING_RATE,
        weight_decay=LINEAR_PROBE_WEIGHT_DECAY,
    )
    criterion = nn.CrossEntropyLoss()

    dataset = TensorDataset(
        torch.from_numpy(train_x),
        torch.from_numpy(train_y.astype(np.int64, copy=False)),
    )
    loader = DataLoader(
        dataset,
        batch_size=min(LINEAR_PROBE_BATCH_SIZE, len(dataset)),
        shuffle=True,
        generator=torch.Generator().manual_seed(RANDOM_SEED + 1),
        num_workers=0,
    )

    best_state = None
    best_epoch = -1
    best_val = -math.inf
    no_improvement = 0
    rows = []

    for epoch in range(1, LINEAR_PROBE_EPOCHS + 1):
        probe.train()
        train_loss_sum = 0.0
        train_count = 0

        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device=device, dtype=torch.float32)
            y_batch = y_batch.to(device=device, dtype=torch.long)
            optimizer.zero_grad(set_to_none=True)
            logits = probe(x_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            train_loss_sum += float(loss.detach()) * len(y_batch)
            train_count += len(y_batch)

        val_metrics = evaluate_linear_probe(
            probe,
            val_x,
            val_y,
            device=device,
        )
        current = float(val_metrics["balanced_accuracy"])
        rows.append(
            {
                "epoch": epoch,
                "train_loss": train_loss_sum / train_count,
                "val_balanced_accuracy": current,
                "val_accuracy": val_metrics["accuracy"],
            }
        )

        if current > best_val + 1e-12:
            best_val = current
            best_epoch = epoch
            best_state = copy.deepcopy(probe.state_dict())
            no_improvement = 0
        else:
            no_improvement += 1

        if no_improvement >= LINEAR_PROBE_PATIENCE:
            break

    if best_state is None:
        raise RuntimeError("Linear probe did not produce a best state")
    probe.load_state_dict(best_state)
    return probe, pd.DataFrame(rows), best_epoch, best_val


probe_raw_train, probe_raw_val, probe_raw_test, probe_mean, probe_std = (
    standardize_feature_splits(
        raw_train["h"],
        raw_val["h"],
        raw_test["h"],
    )
)
probe_recon_test = apply_feature_standardization(
    recon_test["h"],
    probe_mean,
    probe_std,
)

linear_probe, linear_probe_history, linear_probe_best_epoch, linear_probe_best_val = (
    train_linear_probe(
        probe_raw_train,
        raw_train["y"],
        probe_raw_val,
        raw_val["y"],
        num_classes=len(class_to_idx),
        device=DEVICE,
    )
)

raw_linear_probe_metrics = evaluate_linear_probe(
    linear_probe,
    probe_raw_test,
    raw_test["y"],
    device=DEVICE,
)
recon_linear_probe_metrics = evaluate_linear_probe(
    linear_probe,
    probe_recon_test,
    recon_test["y"],
    device=DEVICE,
)

linear_probe_comparison = pd.DataFrame(
    {
        "raw_test": raw_linear_probe_metrics,
        "recon_test_B": recon_linear_probe_metrics,
    }
)
linear_probe_comparison["delta_recon_minus_raw"] = (
    linear_probe_comparison["recon_test_B"] - linear_probe_comparison["raw_test"]
)

print(f"Best raw-validation probe epoch: {linear_probe_best_epoch}")
print(f"Best raw-validation probe BA:    {linear_probe_best_val:.6f}")
display(linear_probe_comparison)

## 13. Unsupervised held-out-user diagnostics: KMeans, NMI, ARI, silhouette

These metrics use only the test embeddings of each condition.

- KMeans is run separately on raw-test and reconstructed-test embeddings with the same seed and `K = number of represented classes`.
- Silhouette uses cosine distance on the L2-normalized embeddings.
- For a fair comparison, raw and reconstruction have identical test labels and sample order.

In [ ]:
def squared_euclidean_matrix(x: np.ndarray, centers: np.ndarray) -> np.ndarray:
    x_sq = np.square(x).sum(axis=1, keepdims=True)
    c_sq = np.square(centers).sum(axis=1, keepdims=True).T
    return np.maximum(x_sq + c_sq - 2.0 * x @ centers.T, 0.0)


def kmeans_plus_plus_init(
    x: np.ndarray,
    k: int,
    *,
    rng: np.random.Generator,
) -> np.ndarray:
    centers = np.empty((k, x.shape[1]), dtype=np.float32)
    first = int(rng.integers(len(x)))
    centers[0] = x[first]
    closest = squared_euclidean_matrix(x, centers[:1])[:, 0]

    for center_index in range(1, k):
        total = float(closest.sum())
        if not math.isfinite(total) or total <= 0:
            chosen = int(rng.integers(len(x)))
        else:
            chosen = int(rng.choice(len(x), p=closest / total))
        centers[center_index] = x[chosen]
        new_distance = squared_euclidean_matrix(
            x,
            centers[center_index : center_index + 1],
        )[:, 0]
        closest = np.minimum(closest, new_distance)
    return centers


def run_kmeans_once(
    x: np.ndarray,
    *,
    k: int,
    rng: np.random.Generator,
    max_iter: int,
) -> tuple[np.ndarray, np.ndarray, float]:
    centers = kmeans_plus_plus_init(x, k, rng=rng)
    previous_assignment = None

    for _ in range(max_iter):
        distances = squared_euclidean_matrix(x, centers)
        assignment = distances.argmin(axis=1).astype(np.int64)
        if previous_assignment is not None and np.array_equal(
            assignment,
            previous_assignment,
        ):
            break
        previous_assignment = assignment.copy()

        nearest_distance = distances[np.arange(len(x)), assignment]
        new_centers = np.empty_like(centers)
        for cluster_index in range(k):
            members = x[assignment == cluster_index]
            if len(members) == 0:
                replacement = int(np.argmax(nearest_distance))
                new_centers[cluster_index] = x[replacement]
                nearest_distance[replacement] = -np.inf
            else:
                new_centers[cluster_index] = members.mean(axis=0)
        centers = new_centers

    final_distances = squared_euclidean_matrix(x, centers)
    assignment = final_distances.argmin(axis=1).astype(np.int64)
    inertia = float(
        final_distances[np.arange(len(x)), assignment].sum()
    )
    return assignment, centers, inertia


def kmeans_best_of_n(
    x: np.ndarray,
    *,
    k: int,
    n_init: int,
    max_iter: int,
    seed: int,
) -> tuple[np.ndarray, np.ndarray, float]:
    best_assignment = None
    best_centers = None
    best_inertia = math.inf

    for init_index in range(n_init):
        rng = np.random.default_rng(seed + init_index)
        assignment, centers, inertia = run_kmeans_once(
            np.asarray(x, dtype=np.float32),
            k=k,
            rng=rng,
            max_iter=max_iter,
        )
        if inertia < best_inertia:
            best_assignment = assignment
            best_centers = centers
            best_inertia = inertia

    if best_assignment is None or best_centers is None:
        raise RuntimeError("KMeans failed")
    return best_assignment, best_centers, best_inertia


def contingency_matrix_numpy(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> np.ndarray:
    _, true_inverse = np.unique(y_true, return_inverse=True)
    _, pred_inverse = np.unique(y_pred, return_inverse=True)
    matrix = np.zeros(
        (true_inverse.max() + 1, pred_inverse.max() + 1),
        dtype=np.int64,
    )
    np.add.at(matrix, (true_inverse, pred_inverse), 1)
    return matrix


def normalized_mutual_information(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> float:
    contingency = contingency_matrix_numpy(y_true, y_pred).astype(np.float64)
    n = contingency.sum()
    row = contingency.sum(axis=1, keepdims=True)
    col = contingency.sum(axis=0, keepdims=True)
    nonzero = contingency > 0
    expected = row @ col / n

    mutual_information = float(
        np.sum(
            (contingency[nonzero] / n)
            * np.log(contingency[nonzero] / expected[nonzero])
        )
    )
    row_prob = row.ravel() / n
    col_prob = col.ravel() / n
    h_true = float(
        -np.sum(row_prob[row_prob > 0] * np.log(row_prob[row_prob > 0]))
    )
    h_pred = float(
        -np.sum(col_prob[col_prob > 0] * np.log(col_prob[col_prob > 0]))
    )
    denominator = h_true + h_pred
    return 1.0 if denominator == 0 else float(
        2.0 * mutual_information / denominator
    )


def comb2(values):
    values = np.asarray(values)
    return values * (values - 1.0) / 2.0


def adjusted_rand_index(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> float:
    contingency = contingency_matrix_numpy(y_true, y_pred).astype(np.float64)
    n = contingency.sum()
    if n < 2:
        return 1.0
    sum_comb = float(comb2(contingency).sum())
    row_comb = float(comb2(contingency.sum(axis=1)).sum())
    col_comb = float(comb2(contingency.sum(axis=0)).sum())
    total_comb = float(comb2(n))
    expected = row_comb * col_comb / total_comb
    maximum = 0.5 * (row_comb + col_comb)
    denominator = maximum - expected
    if denominator == 0:
        return 1.0 if sum_comb == maximum else 0.0
    return float((sum_comb - expected) / denominator)


def cosine_silhouette(
    z: np.ndarray,
    y: np.ndarray,
    *,
    max_samples: int | None,
    seed: int,
) -> tuple[float, float, int]:
    z = np.asarray(z, dtype=np.float64)
    y = np.asarray(y)

    if max_samples is not None and len(z) > max_samples:
        rng = np.random.default_rng(seed)
        indices = np.sort(
            rng.choice(len(z), size=max_samples, replace=False)
        )
    else:
        indices = np.arange(len(z))

    z = z[indices]
    y = y[indices]
    distance = np.clip(1.0 - z @ z.T, 0.0, 2.0)
    np.fill_diagonal(distance, 0.0)

    classes, counts = np.unique(y, return_counts=True)
    class_count = dict(zip(classes.tolist(), counts.tolist()))
    values = []
    labels = []

    for row_index, class_index in enumerate(y):
        if class_count[class_index] < 2:
            continue

        same = y == class_index
        same[row_index] = False
        a = float(distance[row_index, same].mean())

        b = math.inf
        for other_class in classes:
            if other_class == class_index:
                continue
            selected = y == other_class
            b = min(b, float(distance[row_index, selected].mean()))

        denominator = max(a, b)
        values.append(0.0 if denominator <= 1e-12 else (b - a) / denominator)
        labels.append(class_index)

    if not values:
        raise ValueError("No eligible samples for silhouette")

    values = np.asarray(values, dtype=np.float64)
    labels = np.asarray(labels)
    micro = float(values.mean())
    macro = float(
        np.mean(
            [
                values[labels == class_index].mean()
                for class_index in np.unique(labels)
            ]
        )
    )
    return micro, macro, len(values)


def evaluate_unsupervised(
    bundle: dict[str, np.ndarray],
) -> dict[str, float]:
    num_classes = len(np.unique(bundle["y"]))
    assignment, _, inertia = kmeans_best_of_n(
        bundle["z"],
        k=num_classes,
        n_init=KMEANS_N_INIT,
        max_iter=KMEANS_MAX_ITER,
        seed=RANDOM_SEED,
    )
    nmi = normalized_mutual_information(bundle["y"], assignment)
    ari = adjusted_rand_index(bundle["y"], assignment)
    silhouette_micro, silhouette_macro, silhouette_samples = cosine_silhouette(
        bundle["z"],
        bundle["y"],
        max_samples=SILHOUETTE_MAX_SAMPLES,
        seed=RANDOM_SEED,
    )
    return {
        "kmeans_inertia": inertia,
        "NMI_arithmetic": nmi,
        "ARI": ari,
        "silhouette_micro": silhouette_micro,
        "silhouette_macro": silhouette_macro,
        "silhouette_samples": float(silhouette_samples),
    }


raw_unsupervised = evaluate_unsupervised(raw_test)
recon_unsupervised = evaluate_unsupervised(recon_test)

unsupervised_comparison = pd.DataFrame(
    {
        "raw_test": raw_unsupervised,
        "recon_test_B": recon_unsupervised,
    }
)
unsupervised_comparison["delta_recon_minus_raw"] = (
    unsupervised_comparison["recon_test_B"]
    - unsupervised_comparison["raw_test"]
)
display(unsupervised_comparison)

## 14. Common PCA visualization (diagnostic only)

A single PCA basis is fit to the concatenated paired raw/reconstructed test embeddings, so the two plots are directly comparable in the same 2-D coordinate system.

PCA is only a visual diagnostic; use the full 256-D metrics for conclusions.

In [ ]:
def balanced_subsample_indices(
    labels: np.ndarray,
    *,
    max_samples: int,
    seed: int,
) -> np.ndarray:
    if len(labels) <= max_samples:
        return np.arange(len(labels), dtype=np.int64)

    classes = np.unique(labels)
    rng = np.random.default_rng(seed)
    quota = max(1, max_samples // len(classes))
    selected = []
    remaining = []

    for class_index in classes:
        indices = np.flatnonzero(labels == class_index)
        shuffled = rng.permutation(indices)
        selected.extend(shuffled[: min(quota, len(shuffled))].tolist())
        remaining.extend(shuffled[min(quota, len(shuffled)) :].tolist())

    slots = max_samples - len(selected)
    if slots > 0 and remaining:
        selected.extend(
            rng.choice(
                remaining,
                size=min(slots, len(remaining)),
                replace=False,
            ).tolist()
        )
    return np.asarray(sorted(selected[:max_samples]), dtype=np.int64)


pca_indices = balanced_subsample_indices(
    raw_test["y"],
    max_samples=PCA_MAX_SAMPLES,
    seed=RANDOM_SEED,
)
raw_pca_z = np.asarray(raw_test["z"][pca_indices], dtype=np.float64)
recon_pca_z = np.asarray(recon_test["z"][pca_indices], dtype=np.float64)
pca_y = raw_test["y"][pca_indices]

stacked = np.vstack([raw_pca_z, recon_pca_z])
center = stacked.mean(axis=0, keepdims=True)
_, _, vt = np.linalg.svd(stacked - center, full_matrices=False)

raw_xy = (raw_pca_z - center) @ vt[:2].T
recon_xy = (recon_pca_z - center) @ vt[:2].T

for source, coordinates in (
    ("raw_test", raw_xy),
    ("recon_test_B", recon_xy),
):
    plt.figure(figsize=(10, 8))
    for class_index in np.unique(pca_y):
        selected = pca_y == class_index
        plt.scatter(
            coordinates[selected, 0],
            coordinates[selected, 1],
            s=14,
            alpha=0.7,
            label=idx_to_class[int(class_index)],
        )
    plt.xlabel("Common PC1")
    plt.ylabel("Common PC2")
    plt.title(f"{source} — common PCA basis")
    if len(np.unique(pca_y)) <= 20:
        plt.legend(bbox_to_anchor=(1.02, 1.0), loc="upper left")
    plt.tight_layout()
    if SAVE_FIGURES:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        plt.savefig(OUTPUT_DIR / f"{source}_common_pca.png", dpi=180)
    plt.show()

## 15. Final Experiment B comparison table

Interpretation direction:

- `↑`: larger is generally better.
- `↓`: smaller is generally better.
- `context`: not a standalone "higher/lower is better" metric.
- For `D_inter / D_intra`, inspect `D_intra` and `D_inter` individually too.

In [ ]:
def get_same_label(
    table: pd.DataFrame,
    *,
    source: str,
    k: int,
    column: str,
) -> float:
    row = table.loc[(table["source"] == source) & (table["k"] == k)]
    if len(row) != 1:
        raise AssertionError(f"Expected one row for source={source}, k={k}")
    return float(row.iloc[0][column])


raw_metrics = {
    "CNN_test_balanced_accuracy": raw_cnn_eval["metrics"]["balanced_accuracy"],
    "CNN_test_macro_f1": raw_cnn_eval["metrics"]["macro_f1"],
    "kNN_test_balanced_accuracy": raw_retrieval["kNN_balanced_accuracy"],
    "prototype_test_balanced_accuracy": raw_geometry["prototype_balanced_accuracy"],
    "linear_probe_test_balanced_accuracy": raw_linear_probe_metrics["balanced_accuracy"],
    "retrieval_macro_mAP": raw_retrieval["retrieval_mAP_macro"],
    "retrieval_micro_mAP": raw_retrieval["retrieval_mAP_micro"],
    "SameLabel_macro_at_1": get_same_label(
        same_label_comparison,
        source="raw_test",
        k=1,
        column="same_label_macro",
    ),
    "SameLabel_micro_at_1": get_same_label(
        same_label_comparison,
        source="raw_test",
        k=1,
        column="same_label_micro",
    ),
    "D_intra_macro": raw_geometry["D_intra_macro"],
    "D_inter_centroids": raw_geometry["D_inter_centroids"],
    "D_inter_over_D_intra": raw_geometry["D_inter_over_D_intra"],
    "NMI_arithmetic": raw_unsupervised["NMI_arithmetic"],
    "ARI": raw_unsupervised["ARI"],
    "silhouette_macro": raw_unsupervised["silhouette_macro"],
    "silhouette_micro": raw_unsupervised["silhouette_micro"],
}

recon_metrics = {
    "CNN_test_balanced_accuracy": recon_cnn_eval["metrics"]["balanced_accuracy"],
    "CNN_test_macro_f1": recon_cnn_eval["metrics"]["macro_f1"],
    "kNN_test_balanced_accuracy": recon_retrieval["kNN_balanced_accuracy"],
    "prototype_test_balanced_accuracy": recon_geometry["prototype_balanced_accuracy"],
    "linear_probe_test_balanced_accuracy": recon_linear_probe_metrics["balanced_accuracy"],
    "retrieval_macro_mAP": recon_retrieval["retrieval_mAP_macro"],
    "retrieval_micro_mAP": recon_retrieval["retrieval_mAP_micro"],
    "SameLabel_macro_at_1": get_same_label(
        same_label_comparison,
        source="recon_test_B",
        k=1,
        column="same_label_macro",
    ),
    "SameLabel_micro_at_1": get_same_label(
        same_label_comparison,
        source="recon_test_B",
        k=1,
        column="same_label_micro",
    ),
    "D_intra_macro": recon_geometry["D_intra_macro"],
    "D_inter_centroids": recon_geometry["D_inter_centroids"],
    "D_inter_over_D_intra": recon_geometry["D_inter_over_D_intra"],
    "NMI_arithmetic": recon_unsupervised["NMI_arithmetic"],
    "ARI": recon_unsupervised["ARI"],
    "silhouette_macro": recon_unsupervised["silhouette_macro"],
    "silhouette_micro": recon_unsupervised["silhouette_micro"],
}

direction = {
    "CNN_test_balanced_accuracy": "↑",
    "CNN_test_macro_f1": "↑",
    "kNN_test_balanced_accuracy": "↑",
    "prototype_test_balanced_accuracy": "↑",
    "linear_probe_test_balanced_accuracy": "↑",
    "retrieval_macro_mAP": "↑",
    "retrieval_micro_mAP": "↑",
    "SameLabel_macro_at_1": "↑",
    "SameLabel_micro_at_1": "↑",
    "D_intra_macro": "↓",
    "D_inter_centroids": "↑ / context",
    "D_inter_over_D_intra": "↑",
    "NMI_arithmetic": "↑",
    "ARI": "↑",
    "silhouette_macro": "↑",
    "silhouette_micro": "↑",
}

final_comparison = pd.DataFrame(
    {
        "direction": pd.Series(direction),
        "raw_test": pd.Series(raw_metrics),
        "recon_test_B": pd.Series(recon_metrics),
    }
)
final_comparison["delta_recon_minus_raw"] = (
    final_comparison["recon_test_B"] - final_comparison["raw_test"]
)

display(final_comparison)
print()
print(
    "Paired mean embedding cosine similarity:",
    f"{paired_summary.iloc[0]['mean_embedding_cosine_similarity']:.6f}",
)
print(
    "Prediction agreement:",
    f"{paired_summary.iloc[0]['prediction_agreement']:.6f}",
)

## 16. Export Experiment B artifacts and provenance

In [ ]:
def json_safe(value: object) -> object:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, torch.device):
        return str(value)
    if isinstance(value, Mapping):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(item) for item in value]
    return value


if SAVE_ARTIFACTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    final_comparison.to_csv(
        OUTPUT_DIR / "experiment_B_raw_vs_reconstruction_metrics.csv"
    )
    classification_comparison.to_csv(
        OUTPUT_DIR / "classification_comparison.csv"
    )
    retrieval_comparison.to_csv(
        OUTPUT_DIR / "retrieval_comparison.csv"
    )
    same_label_comparison.to_csv(
        OUTPUT_DIR / "same_label_at_k_comparison.csv",
        index=False,
    )
    geometry_comparison.to_csv(
        OUTPUT_DIR / "geometry_comparison.csv"
    )
    intra_by_class.to_csv(
        OUTPUT_DIR / "intra_distance_by_class_raw_vs_recon.csv",
        index=False,
    )
    linear_probe_comparison.to_csv(
        OUTPUT_DIR / "linear_probe_comparison.csv"
    )
    unsupervised_comparison.to_csv(
        OUTPUT_DIR / "unsupervised_comparison.csv"
    )
    paired_summary.to_csv(
        OUTPUT_DIR / "paired_preservation_summary.csv",
        index=False,
    )
    transition_table.to_csv(
        OUTPUT_DIR / "paired_prediction_transitions.csv",
        index=False,
    )
    paired_per_class.to_csv(
        OUTPUT_DIR / "paired_preservation_by_class.csv",
        index=False,
    )
    validation_knn_table.to_csv(
        OUTPUT_DIR / "raw_validation_knn_k_selection.csv",
        index=False,
    )
    linear_probe_history.to_csv(
        OUTPUT_DIR / "raw_linear_probe_history.csv",
        index=False,
    )

    np.savez_compressed(
        OUTPUT_DIR / "reconstructed_test_embeddings.npz",
        h=recon_test["h"],
        z=recon_test["z"],
        y=recon_test["y"],
        cnn_pred=recon_test["cnn_pred"],
        logits=recon_test["logits"],
        sample_id=recon_test["sample_id"],
        paired_raw_embedding_cosine_similarity=paired_cosine_similarity,
        paired_raw_embedding_cosine_distance=paired_cosine_distance,
    )

    test_manifest = sample_manifest.loc[
        sample_manifest["split"] == "test"
    ].copy().reset_index(drop=True)
    if test_manifest["sample_id"].tolist() != recon_test["sample_id"].tolist():
        raise AssertionError("Export manifest order does not match recon embeddings")
    test_manifest["raw_cnn_prediction_idx"] = raw_test["cnn_pred"]
    test_manifest["recon_cnn_prediction_idx"] = recon_test["cnn_pred"]
    test_manifest["raw_cnn_prediction_label"] = [
        idx_to_class[int(i)] for i in raw_test["cnn_pred"]
    ]
    test_manifest["recon_cnn_prediction_label"] = [
        idx_to_class[int(i)] for i in recon_test["cnn_pred"]
    ]
    test_manifest["raw_correct"] = raw_correct
    test_manifest["recon_correct"] = recon_correct
    test_manifest["prediction_transition"] = transition_labels
    test_manifest["paired_embedding_cosine_similarity"] = paired_cosine_similarity
    test_manifest["paired_embedding_cosine_distance"] = paired_cosine_distance
    test_manifest.to_csv(
        OUTPUT_DIR / "paired_test_manifest.csv",
        index=False,
    )

    provenance = {
        "experiment": "B",
        "protocol": (
            "raw-trained frozen CNN; raw-train normalization; "
            "reconstructed held-out test input"
        ),
        "repository_root": REPOSITORY_ROOT,
        "padded_root": PADDED_ROOT,
        "baseline_checkpoint": BASELINE_CHECKPOINT_PATH,
        "baseline_checkpoint_best_epoch": checkpoint.get("best_epoch"),
        "baseline_checkpoint_best_val_balanced_accuracy": checkpoint.get(
            "best_val_balanced_accuracy"
        ),
        "producer_metadata": asdict(producer_metadata),
        "class_to_idx": class_to_idx,
        "train_users": TRAIN_USERS,
        "val_users": VAL_USERS,
        "test_users": TEST_USERS,
        "normalization": {
            "source": "loaded from raw-trained baseline checkpoint",
            "mean": acceleration_mean,
            "std": acceleration_std,
        },
        "reconstruction_suffix": RECONSTRUCTION_SUFFIX,
        "retrieval_protocol": {
            "gallery": "raw train embeddings",
            "k_selection": "raw validation queries against raw train gallery",
            "test_queries": ["raw test", "reconstructed test"],
            "selected_k": selected_knn_k,
        },
        "prototype_protocol": "centroids from raw train embeddings",
        "linear_probe_protocol": (
            "probe trained on raw train h, selected on raw val h, "
            "applied unchanged to raw/reconstructed test h"
        ),
        "final_comparison": final_comparison.reset_index().rename(
            columns={"index": "metric"}
        ).to_dict(orient="records"),
        "paired_summary": paired_summary.iloc[0].to_dict(),
        "prediction_transitions": transition_table.to_dict(orient="records"),
    }
    (OUTPUT_DIR / "experiment_B_provenance.json").write_text(
        json.dumps(json_safe(provenance), ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )

    print(f"Artifacts written to: {OUTPUT_DIR}")
    display(
        pd.DataFrame(
            {"artifact": sorted(path.name for path in OUTPUT_DIR.iterdir())}
        )
    )
else:
    print("SAVE_ARTIFACTS=False; no files were written.")

## Interpretation checklist

Read the result in this order:

1. **Direct preservation:** compare frozen-CNN `balanced_accuracy`, `macro_f1`, prediction agreement, and paired raw↔reconstruction embedding cosine similarity.
2. **Local / retrieval structure:** compare kNN BA, SameLabel@K and mAP, always against the unchanged raw-train gallery.
3. **Class geometry:** compare `D_intra`, `D_inter`, `D_inter/D_intra`, and silhouette.
4. **Global label structure:** compare prototype BA, raw-trained linear-probe BA, NMI and ARI.

A reconstruction looks structurally better when several indicators agree, for example:

- `D_intra ↓`,
- `D_inter` maintained or increased,
- `D_inter/D_intra ↑`,
- silhouette ↑,
- SameLabel@K / mAP / kNN / prototype ↑.

Do **not** conclude "information was destroyed" from frozen-CNN accuracy alone. A large accuracy drop with relatively preserved geometry can still indicate a systematic raw→reconstruction domain shift. Experiment C (train and test on reconstruction) is the clean follow-up for separating that possibility.